# KLUE-RoBERTa 기반 한국어 문서 질의응답

## 프로젝트 목표

사용자가 문단과 질문을 입력하면 **문단 안의 답변 구절**을 추출하고 원문에서 강조 표시합니다. 이 노트북은 과제 3의 모델·데이터 소개, Tokenizer 전처리, fine-tuning, 평가, 직접 입력 예측, 오류 분석을 순서대로 수행합니다.

- 사전학습 모델: [klue/roberta-base](https://huggingface.co/klue/roberta-base)
- 학습 데이터: [KorQuAD 1.0](https://huggingface.co/datasets/KorQuAD/squad_kor_v1)
- 문제 유형: 추출형 질의응답. 답은 제공된 문단에 실제로 포함된 연속된 구절이어야 합니다.
- 데이터: 학습 60,407개, 공개 검증 5,774개. 공개 테스트 정답은 제공되지 않습니다.
- 모델 선택: 학습 데이터에서 기사 제목 단위로 나눈 내부 검증 세트의 loss
- 최종 보고 점수: 공개 검증 세트의 정규화 EM과 글자 단위 F1. 공식 리더보드 점수와 직접 비교하지 않습니다.

**실행 방법:** 설치 셀부터 마지막 셀까지 순서대로 실행합니다. 원본 학습 자료를 기사 제목 단위로 내부 학습·검증에 나누고, 내부 학습 자료 중 20%를 seed 42로 뽑아 5 epoch 학습한 뒤 공개 검증 5,774개로 최종 평가합니다.

**제출 파일:** 이 노트북 하나에 데이터 확인, 실제 학습 로그, 학습·검증 곡선, 평가 점수, 오류 분석, 직접 입력 예측과 최종 보고서가 모두 표시됩니다. 별도의 실행 점검 노트북이나 실행용 Python 파일은 사용하지 않습니다. 완료 후 출력이 포함된 이 `.ipynb`를 저장해 제출하세요. Colab에서는 `파일 → 다운로드 → .ipynb 다운로드`를 선택합니다.

**모델 보관:** 재사용할 모델과 학습 재개용 체크포인트는 `outputs/korquad_qa` 아래에 저장됩니다. 결과를 읽는 데에는 이 노트북만 있으면 됩니다.

**이번 실험 규모:** 내부 학습 약 10,906개(학습용 분할의 20%), 내부 검증 5,876개, 공개 검증 5,774개. 학습 자료 전체를 사용한 결과로 표기하지 않습니다.

**현재 상태:** 학습을 실행하지 않은 수정본입니다. 아래 셀들을 실행하면 학습 자료 20%로 5 epoch 학습하고, 평가·분석 결과가 이 노트북 안에 표시됩니다.


## 0. 환경 준비

첫 실행 시 설치합니다. GPU가 있는 Colab 또는 Jupyter 환경을 권장합니다. 패키지 설치 후 커널 재시작이 필요할 수 있습니다.

In [ ]:
%pip install -q "torch" "transformers>=4.46,<5" "datasets>=3.0,<5" "accelerate>=1.0,<2" "scikit-learn>=1.4,<2" "numpy>=1.26" "pandas>=2.0" "matplotlib>=3.8"


## 1. 설정과 재현성

학습용 분할의 20%를 사용해 5 epoch 학습합니다. 검증 자료는 줄이지 않으며 실행 장치·패키지 버전·표본 추출 seed를 기록합니다.


In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from importlib.metadata import version
import platform
import time
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, Markdown, display
from sklearn.model_selection import GroupShuffleSplit
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    DefaultDataCollator,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

SEED = 42
TRAIN_FRACTION = 0.2
NUM_EPOCHS = 5
assert 0 < TRAIN_FRACTION <= 1
MODEL_NAME = "klue/roberta-base"
DATASET_NAME = "KorQuAD/squad_kor_v1"
MAX_LENGTH = 384
DOC_STRIDE = 128
N_BEST_SIZE = 20
MAX_ANSWER_LENGTH = 40
RUN_ID = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S_%f")
RUN_MODE = f"train{int(TRAIN_FRACTION * 100)}_epochs{NUM_EPOCHS}"
OUTPUT_DIR = Path("outputs/korquad_qa") / f"{RUN_MODE}_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
print("PyTorch:", torch.__version__, "| CUDA 사용:", torch.cuda.is_available())
device_name = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
run_config = {
    "run_id": RUN_ID, "mode": RUN_MODE, "seed": SEED,
    "model": MODEL_NAME, "dataset": DATASET_NAME,
    "max_length": MAX_LENGTH, "doc_stride": DOC_STRIDE,
    "n_best_size": N_BEST_SIZE, "max_answer_length": MAX_ANSWER_LENGTH,
    "epochs": NUM_EPOCHS, "train_fraction": TRAIN_FRACTION,
    "device": device_name, "python": platform.python_version(),
    "packages": {name: version(name) for name in [
        "torch", "transformers", "datasets", "accelerate", "scikit-learn",
        "numpy", "pandas", "matplotlib",
    ]},
}
print("실행 모드:", RUN_MODE, "| 장치:", device_name)
print("결과 폴더:", OUTPUT_DIR.resolve())
if device_name == "cpu":
    print("학습은 CPU에서 오래 걸립니다. Colab GPU 사용을 권장합니다.")

# 학습을 중단한 뒤 재개할 때만 체크포인트 경로를 지정합니다.
RESUME_FROM_CHECKPOINT = None


## 2. 데이터 확인

답변 시작 위치가 실제 문단의 답변 문자열을 가리키는지 확인합니다. 같은 기사에 여러 질문이 있을 수 있으므로 내부 학습·검증 분리는 제목별로 진행합니다.

In [ ]:
dataset = load_dataset(DATASET_NAME)
print(dataset)
print("열:", dataset["train"].column_names)
example = dataset["train"][0]
display({
    "title": example["title"],
    "question": example["question"],
    "answer": example["answers"]["text"][0],
    "answer_start": example["answers"]["answer_start"][0],
})
assert {"id", "title", "context", "question", "answers"}.issubset(dataset["train"].column_names)

def answer_span_matches(row):
    answers = row["answers"]
    if not answers["text"] or not answers["answer_start"]:
        return False
    start = answers["answer_start"][0]
    text = answers["text"][0]
    return row["context"][start:start + len(text)] == text

sample_check = dataset["train"].select(range(min(1000, len(dataset["train"]))))
print("처음 1000개 정답 위치 일치 비율:",
      round(np.mean([answer_span_matches(row) for row in sample_check]), 4))
print("학습 문단 길이 기술통계:")
display(pd.Series([len(text) for text in dataset["train"]["context"][:2000]]).describe())

In [ ]:
all_train = dataset["train"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
train_idx, select_idx = next(
    splitter.split(np.zeros(len(all_train)), groups=all_train["title"])
)
train_examples = all_train.select(train_idx)
selection_examples = all_train.select(select_idx)
final_examples = dataset["validation"]
assert set(train_examples["title"]).isdisjoint(set(selection_examples["title"]))

# 내부 학습 분할에서만 무작위 표본을 뽑습니다. 내부·공개 검증은 모두 유지합니다.
source_train_size = len(train_examples)
sample_size = max(1, int(source_train_size * TRAIN_FRACTION))
train_examples = train_examples.shuffle(seed=SEED).select(range(sample_size))
print("원래 내부 학습:", source_train_size, "→ 사용:", len(train_examples),
      f"({TRAIN_FRACTION:.0%}), epoch: {NUM_EPOCHS}")
run_config["source_train_examples"] = source_train_size
run_config["sampled_train_examples"] = len(train_examples)

print("내부 학습:", len(train_examples),
      "| 내부 검증:", len(selection_examples),
      "| 공개 검증:", len(final_examples))
run_config["examples"] = {
    "train": len(train_examples), "selection": len(selection_examples),
    "public_validation": len(final_examples),
}
with (OUTPUT_DIR / "run_config.json").open("w", encoding="utf-8") as handle:
    json.dump(run_config, handle, ensure_ascii=False, indent=2)


## 3. Tokenizer와 정답 위치 변환

질문과 문단을 함께 토큰화합니다. 긴 문단은 MAX_LENGTH 길이의 창으로 나누고 DOC_STRIDE만큼 겹칩니다. 원본 정답의 문자 위치를 각 창의 시작·끝 토큰 위치로 변환합니다. 정답이 창 바깥에 있으면 CLS 토큰을 정답 위치로 사용합니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast, "정답 위치 변환에는 fast tokenizer가 필요합니다."

def prepare_train_features(examples):
    encoded = tokenizer(
        [q.strip() for q in examples["question"]],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_token_type_ids=False,  # RoBERTa는 segment ID 0만 사용합니다.
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = encoded.pop("overflow_to_sample_mapping")
    offset_maps = encoded.pop("offset_mapping")
    start_positions, end_positions = [], []

    for feature_idx, offsets in enumerate(offset_maps):
        sample_idx = sample_map[feature_idx]
        answers = examples["answers"][sample_idx]
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        sequence_ids = encoded.sequence_ids(feature_idx)
        cls_index = encoded["input_ids"][feature_idx].index(tokenizer.cls_token_id)
        context_tokens = [i for i, sid in enumerate(sequence_ids) if sid == 1]

        if not context_tokens:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        context_start, context_end = context_tokens[0], context_tokens[-1]
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        token_start = context_start
        while token_start <= context_end and offsets[token_start][0] <= start_char:
            token_start += 1
        token_end = context_end
        while token_end >= context_start and offsets[token_end][1] >= end_char:
            token_end -= 1
        start_positions.append(token_start - 1)
        end_positions.append(token_end + 1)

    encoded["start_positions"] = start_positions
    encoded["end_positions"] = end_positions
    return encoded

train_features = train_examples.map(
    prepare_train_features, batched=True, remove_columns=train_examples.column_names
)
selection_features = selection_examples.map(
    prepare_train_features, batched=True, remove_columns=selection_examples.column_names
)
print("학습 예제:", len(train_examples), "→ 토큰 창:", len(train_features))
print("첫 창의 정답 토큰 위치:",
      train_features[0]["start_positions"], train_features[0]["end_positions"])

## 4. 공개 검증 세트 전처리

평가용으로 각 토큰 창의 원래 문단 위치를 보관합니다. 모델 예측 후 이 위치를 이용해 답변 문자열을 복원합니다.

In [ ]:
def prepare_eval_features(examples):
    encoded = tokenizer(
        [q.strip() for q in examples["question"]],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_token_type_ids=False,  # RoBERTa는 segment ID 0만 사용합니다.
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = encoded.pop("overflow_to_sample_mapping")
    encoded["example_id"] = []
    for feature_idx, sample_idx in enumerate(sample_map):
        encoded["example_id"].append(examples["id"][sample_idx])
        sequence_ids = encoded.sequence_ids(feature_idx)
        encoded["offset_mapping"][feature_idx] = [
            offset if sequence_ids[i] == 1 else None
            for i, offset in enumerate(encoded["offset_mapping"][feature_idx])
        ]
    return encoded

final_features = final_examples.map(
    prepare_eval_features, batched=True, remove_columns=final_examples.column_names
)
print("공개 검증 예제:", len(final_examples), "→ 토큰 창:", len(final_features))
assert set(final_features["example_id"]) == set(final_examples["id"])

## 5. 사전학습 모델 Fine-tuning

KLUE-RoBERTa에 질의응답 헤드를 붙여 정답의 시작·끝 토큰을 학습합니다. 500 step마다 내부 검증과 체크포인트 저장을 수행하고, 내부 검증 loss가 가장 낮은 체크포인트의 모델을 사용합니다. 학습용 분할의 20%를 사용해 총 5 epoch 학습합니다.

In [ ]:
class TrainingProgress(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.monotonic()

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step == 1 or state.global_step % 50 == 0:
            elapsed = time.monotonic() - self.started_at
            progress = {
                "status": "training", "step": state.global_step,
                "total_steps": state.max_steps, "epoch": state.epoch,
                "elapsed_seconds": round(elapsed, 1),
                "estimated_remaining_seconds": round(
                    elapsed / state.global_step * (state.max_steps - state.global_step), 1
                ),
            }
            with (OUTPUT_DIR / "progress.json").open("w", encoding="utf-8") as handle:
                json.dump(progress, handle, ensure_ascii=False, indent=2)
            print("학습 진행:", progress, flush=True)

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=8 if torch.cuda.is_available() else 2,
    per_device_eval_batch_size=16 if torch.cuda.is_available() else 4,
    gradient_accumulation_steps=2 if torch.cuda.is_available() else 8,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,
    disable_tqdm=True,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_features,
    eval_dataset=selection_features,
    processing_class=tokenizer,
    data_collator=DefaultDataCollator(),
    callbacks=[TrainingProgress()],
)
train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
print("학습 결과:", train_result.metrics)
trainer.state.save_to_json(str(OUTPUT_DIR / "trainer_state.json"))
with (OUTPUT_DIR / "train_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(train_result.metrics, handle, ensure_ascii=False, indent=2)


In [ ]:
history = pd.DataFrame(trainer.state.log_history)
display(history.tail(10))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, title in zip(
    axes, ["loss", "eval_loss"], ["Training loss", "Internal validation loss"]
):
    if column in history:
        points = history.dropna(subset=[column, "epoch"])
        ax.plot(points["epoch"], points[column], marker="o")
    ax.set(xlabel="Epoch", ylabel="Loss", title=title)
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## 6. 토큰 예측을 답변 구절로 복원

각 토큰 창에서 가능성 높은 시작·끝 위치를 조합합니다. 문단 토큰만 후보로 사용하고, 답변 길이 제한을 적용한 뒤 가장 점수가 높은 구절을 선택합니다. 여기의 점수는 로짓의 합이며 확률로 해석하지 않습니다.

In [ ]:
def postprocess_qa(examples, features, start_logits, end_logits,
                   n_best_size=N_BEST_SIZE, max_answer_length=MAX_ANSWER_LENGTH):
    features_by_id = defaultdict(list)
    for feature_idx, feature in enumerate(features):
        features_by_id[feature["example_id"]].append(feature_idx)

    predictions = {}
    for example in examples:
        context = example["context"]
        best_answer, best_score, best_span = "", -float("inf"), None
        for feature_idx in features_by_id[example["id"]]:
            offsets = features[feature_idx]["offset_mapping"]
            starts = np.argsort(start_logits[feature_idx])[-n_best_size:][::-1]
            ends = np.argsort(end_logits[feature_idx])[-n_best_size:][::-1]
            for start_idx in starts:
                for end_idx in ends:
                    if start_idx >= len(offsets) or end_idx >= len(offsets):
                        continue
                    if offsets[start_idx] is None or offsets[end_idx] is None:
                        continue
                    if end_idx < start_idx or end_idx - start_idx + 1 > max_answer_length:
                        continue
                    char_start = offsets[start_idx][0]
                    char_end = offsets[end_idx][1]
                    if not (0 <= char_start < char_end <= len(context)):
                        continue
                    score = float(start_logits[feature_idx][start_idx]
                                  + end_logits[feature_idx][end_idx])
                    if score > best_score:
                        best_answer = context[char_start:char_end]
                        best_score = score
                        best_span = (char_start, char_end)
        predictions[example["id"]] = {
            "answer": best_answer, "span": best_span, "score": best_score
        }
    return predictions

## 7. 공개 검증 세트 평가

공개 검증 세트는 모델 선택에 사용하지 않았습니다. 아래 EM은 정규화한 예측과 정답이 완전히 같은 비율이고, 글자 F1은 정규화한 글자의 겹침입니다. 두 값은 이 노트북의 자체 지표이므로 공식 KorQuAD 리더보드 점수라고 표기하지 않습니다.

In [ ]:
prediction_inputs = final_features.remove_columns(["example_id", "offset_mapping"])
prediction_output = trainer.predict(prediction_inputs)
start_logits, end_logits = prediction_output.predictions[:2]
predictions = postprocess_qa(
    final_examples, final_features, start_logits, end_logits
)

def normalize_answer(text):
    text = str(text).lower()
    text = "".join(
        char for char in text
        if not unicodedata.category(char).startswith("P")
    )
    return re.sub(r"\s+", "", text)

def char_f1(prediction, truth):
    p = list(normalize_answer(prediction))
    t = list(normalize_answer(truth))
    if not p or not t:
        return float(p == t)
    overlap = sum((Counter(p) & Counter(t)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(t)
    return 2 * precision * recall / (precision + recall)

def score_one(prediction, gold_answers):
    return (
        max(float(normalize_answer(prediction) == normalize_answer(gold))
            for gold in gold_answers),
        max(char_f1(prediction, gold) for gold in gold_answers),
    )

# 실제 정답이 현재 창 크기와 답변 길이 제한으로 복원 가능한지도 확인합니다.
features_by_example = defaultdict(list)
for feature in final_features:
    features_by_example[feature["example_id"]].append(feature)

def gold_decoding_status(example):
    window_contains_gold = False
    for feature in features_by_example[example["id"]]:
        offsets = [offset for offset in feature["offset_mapping"] if offset is not None]
        if not offsets:
            continue
        for start, text in zip(example["answers"]["answer_start"],
                               example["answers"]["text"]):
            end = start + len(text)
            if offsets[0][0] <= start and offsets[-1][1] >= end:
                window_contains_gold = True
                token_count = sum(a < end and b > start for a, b in offsets)
                if 0 < token_count <= MAX_ANSWER_LENGTH:
                    return "복원 가능"
    return "답변 길이 제한 초과" if window_contains_gold else "정답 전체를 포함하는 창 없음"

rows = []
for example in final_examples:
    predicted = predictions[example["id"]]["answer"]
    golds = example["answers"]["text"]
    em, f1 = score_one(predicted, golds)
    rows.append({
        "id": example["id"], "title": example["title"],
        "question": example["question"], "prediction": predicted,
        "gold": golds[0], "gold_answers": golds,
        "context": example["context"],
        "prediction_span": predictions[example["id"]]["span"],
        "window_count": len(features_by_example[example["id"]]),
        "gold_decoding_status": gold_decoding_status(example),
        "em": em, "char_f1": f1,
        "context_length": len(example["context"]),
    })
result_df = pd.DataFrame(rows)
metrics = {
    "normalized_em": float(result_df["em"].mean() * 100),
    "character_f1": float(result_df["char_f1"].mean() * 100),
    "examples": len(result_df),
    "training_mode": RUN_MODE,
    "train_fraction": TRAIN_FRACTION,
    "train_examples": len(train_examples),
    "epochs": NUM_EPOCHS,
}
print(metrics)
display(result_df.head(10))
with (OUTPUT_DIR / "metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, ensure_ascii=False, indent=2)


## 8. 실제 예측에 기반한 오류 분석

문단 길이별 성능과 오답 유형을 함께 비교합니다. 표의 EM·글자 F1은 모두 **0~100** 단위입니다.
오답은 정답 경계 차이, 일부 글자 겹침, 글자 겹침 없음, 빈 예측, 전처리·길이 제약으로 분류합니다.
분류는 관찰 가능한 현상을 표시하는 규칙이며, 모델이 틀린 인과적 원인을 확정하지 않습니다.
대표 사례에는 질문·정답·예측과 실제 원문 일부를 함께 표시합니다.


In [ ]:
length_bins = pd.cut(
    result_df["context_length"],
    bins=[0, 500, 1000, 2000, float("inf")],
    labels=["500자 이하", "501~1000자", "1001~2000자", "2000자 초과"],
)
by_length = result_df.groupby(length_bins, observed=True).agg(
    건수=("id", "count"), EM=("em", "mean"), 글자_F1=("char_f1", "mean"),
)
by_length[["EM", "글자_F1"]] *= 100
display(by_length)

def classify_error(row):
    if row["em"] == 1:
        return "정답"
    if row["gold_decoding_status"] != "복원 가능":
        return row["gold_decoding_status"]
    prediction = normalize_answer(row["prediction"])
    if not prediction:
        return "빈 예측"
    if any(prediction in normalize_answer(gold) or normalize_answer(gold) in prediction
           for gold in row["gold_answers"] if normalize_answer(gold)):
        return "정답 경계 차이"
    return "일부 글자 겹침" if row["char_f1"] > 0 else "글자 겹침 없음"

result_df["error_type"] = result_df.apply(classify_error, axis=1)
error_df = result_df.loc[result_df["em"] == 0].copy()
error_summary = error_df.groupby("error_type").agg(
    건수=("id", "count"), 평균_글자_F1=("char_f1", "mean"),
).sort_values("건수", ascending=False)
error_summary["평균_글자_F1"] *= 100
error_summary["전체_평가_대비_비율"] = error_summary["건수"] / len(result_df) * 100
display(error_summary)

ERROR_GUIDANCE = {
    "정답 경계 차이": (
        "예측과 정답 중 한쪽이 다른 쪽에 포함됩니다. 핵심 표현을 찾았더라도 조사·수식어·구절 경계가 다를 수 있습니다.",
        "실제 원문 경계를 확인하고, 답변 길이 제한과 후보 수를 내부 검증 세트에서 비교합니다.",
    ),
    "일부 글자 겹침": (
        "정답과 일부 글자는 겹치지만 포함 관계는 아닙니다. 글자 F1만으로 의미상 부분 정답이라고 판단할 수 없습니다.",
        "질문의 조건과 선택한 구절을 대조해 부분 추출인지 다른 대상을 고른 것인지 사람이 확인합니다.",
    ),
    "글자 겹침 없음": (
        "정규화한 예측과 정답에 공통 글자가 없습니다. 다른 인물·날짜·수치 또는 무관한 구절을 선택했는지 확인해야 합니다.",
        "원문에서 질문의 조건과 답변 근거를 확인하고, 질문 유형별 오류를 추가 집계합니다.",
    ),
    "빈 예측": (
        "후처리에서 유효한 답변 후보를 찾지 못했습니다. 답이 없다고 판정한 결과는 아닙니다.",
        "문단 토큰 마스킹, 후보 수, offset과 구절 길이 조건을 점검합니다.",
    ),
    "답변 길이 제한 초과": (
        "정답을 포함하는 창은 있지만 정답의 토큰 수가 현재 답변 길이 제한을 넘습니다.",
        "학습 자료의 정답 토큰 길이 분포를 확인하고 내부 검증에서 MAX_ANSWER_LENGTH를 조정합니다.",
    ),
    "정답 전체를 포함하는 창 없음": (
        "현재 전처리의 어떤 창도 정답 구절 전체를 포함하지 않습니다.",
        "정답 위치의 정확성과 MAX_LENGTH·DOC_STRIDE를 확인하고 내부 검증에서 창 설정을 비교합니다.",
    ),
}

import html

def context_excerpt(row, radius=100):
    context = row["context"]
    span = row["prediction_span"]
    start = span[0] if span is not None else context.find(row["gold"])
    start = max(start, 0)
    stop = span[1] if span is not None else start + len(row["gold"])
    left, right = max(0, start - radius), min(len(context), stop + radius)
    prefix, suffix = ("…" if left else ""), ("…" if right < len(context) else "")
    if span is None:
        return prefix + html.escape(context[left:right]) + suffix
    return (prefix + html.escape(context[left:span[0]]) + "<mark>"
            + html.escape(context[span[0]:span[1]]) + "</mark>"
            + html.escape(context[span[1]:right]) + suffix)

case_blocks = []
for error_type, group in error_df.groupby("error_type", sort=True):
    observation, improvement = ERROR_GUIDANCE[error_type]
    case_blocks.append(f"### {error_type}\n\n{observation}\n\n개선 방향: {improvement}\n")
    for _, row in group.sort_values(["char_f1", "id"]).head(2).iterrows():
        block = (
            f"<p><b>사례 ID:</b> {html.escape(row['id'])}<br>"
            f"<b>질문:</b> {html.escape(row['question'])}<br>"
            f"<b>정답:</b> {html.escape(' / '.join(row['gold_answers']))}<br>"
            f"<b>예측:</b> {html.escape(row['prediction']) or '(빈 문자열)'}<br>"
            f"<b>글자 F1:</b> {row['char_f1'] * 100:.2f}<br>"
            f"<b>문단 길이:</b> {row['context_length']}자 / 창 {row['window_count']}개</p>\n\n"
            f"<blockquote>{context_excerpt(row)}</blockquote>\n"
        )
        case_blocks.append(block)
case_analysis_markdown = "\n".join(case_blocks) or "평가한 범위에서 EM 기준 오답이 없습니다."
display(Markdown(case_analysis_markdown))


## 9. 새로운 문단과 질문 직접 입력

다음 함수는 긴 문단에도 토큰 창을 적용합니다. 결과의 답변 위치를 사용해 원문 안의 근거를 강조합니다. 직접 작성한 예시 문단으로 동작을 확인하고 질문·문단을 자유롭게 바꿔보세요.

In [ ]:
def answer_question(question, context, model=trainer.model,
                    tokenizer=tokenizer):
    if not question.strip() or not context.strip():
        raise ValueError("질문과 문단을 모두 입력하세요.")
    encoded = tokenizer(
        question.strip(), context,
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_token_type_ids=False,  # RoBERTa는 segment ID 0만 사용합니다.
        return_offsets_mapping=True,
        padding="max_length",
        return_tensors="pt",
    )
    sequence_ids = [encoded.sequence_ids(i)
                    for i in range(encoded["input_ids"].shape[0])]
    offsets = encoded.pop("offset_mapping").tolist()
    encoded.pop("overflow_to_sample_mapping", None)
    device = next(model.parameters()).device
    model_inputs = {key: value.to(device) for key, value in encoded.items()}
    model.eval()
    with torch.no_grad():
        output = model(**model_inputs)
    start_scores = output.start_logits.cpu().numpy()
    end_scores = output.end_logits.cpu().numpy()

    features = []
    for window_idx, window_offsets in enumerate(offsets):
        features.append({
            "example_id": "user_input",
            "offset_mapping": [
                offset if sequence_ids[window_idx][token_idx] == 1 else None
                for token_idx, offset in enumerate(window_offsets)
            ],
        })
    example = [{"id": "user_input", "context": context}]
    return postprocess_qa(example, features, start_scores, end_scores)["user_input"]

def show_highlight(context, span):
    if span is None:
        print("답변 구절을 찾지 못했습니다.")
        return
    import html
    left = html.escape(context[:span[0]])
    middle = html.escape(context[span[0]:span[1]])
    right = html.escape(context[span[1]:])
    display(HTML(left + "<mark>" + middle + "</mark>" + right))

new_context = (
    "도서관은 평일 오전 9시부터 오후 8시까지 운영한다. "
    "토요일에는 오전 10시부터 오후 5시까지 운영하며, 일요일과 공휴일에는 휴관한다."
)
new_question = "토요일 도서관 운영 시간은 언제인가요?"
answer = answer_question(new_question, new_context)
print("질문:", new_question)
print("예측 답변:", answer["answer"])
show_highlight(new_context, answer["span"])
demo_result = {
    "question": new_question, "context": new_context,
    "expected_answer": "오전 10시부터 오후 5시까지",
    "prediction": answer["answer"], "span": answer["span"],
}
demo_em, demo_f1 = score_one(demo_result["prediction"], [demo_result["expected_answer"]])
demo_result.update(normalized_em=demo_em * 100, character_f1=demo_f1 * 100)
with (OUTPUT_DIR / "demo_prediction.json").open("w", encoding="utf-8") as handle:
    json.dump(demo_result, handle, ensure_ascii=False, indent=2)


## 10. 학습한 모델 저장

내부 검증으로 선택한 최적 모델과 Tokenizer를 저장합니다. 평가 표, 곡선과 분석 보고서는 모두 이 노트북의 출력에도 표시되므로 별도 보고서 파일을 열 필요가 없습니다.


In [ ]:
model_dir = OUTPUT_DIR / "model"
trainer.save_model(str(model_dir))
tokenizer.save_pretrained(str(model_dir))
print("최적 모델 저장:", model_dir.resolve())
print("학습·평가·분석 결과는 이 노트북의 출력에 포함됩니다.")


## 11. 실험 결과와 개선 방향

아래 셀은 이번 실행의 실제 지표·학습 이력·오답·직접 입력 결과를 연결한 분석 보고서를 생성합니다.
자동 분류는 오답 현상을 정리하는 도구이므로,
원인에 대한 최종 판단에는 원문과 질문을 읽는 검토가 필요합니다.

### 모델과 데이터의 연결

KLUE-RoBERTa는 문맥을 양방향으로 읽는 encoder 계열 사전학습 모델입니다. 질문과 문단의 각 토큰을
문맥에 맞게 표현하고, 질의응답 헤드가 정답의 시작·끝 토큰을 예측합니다. 이 과제의 출력은 새 문장 생성이
아니라 원문 구절의 선택이므로 이러한 구조와 잘 맞습니다. QA 헤드는 fine-tuning으로 학습해야 합니다.

### 오류 관찰과 개선 실험

- **경계 차이:** 정답을 너무 짧거나 길게 추출하는지 원문으로 확인합니다. 후보 수·답변 길이 제한 변경은 내부 검증에서 비교합니다.
- **잘못된 대상 선택:** 질문의 인물·시간·장소 조건을 놓쳤는지 검토하고 질문 유형별 평가를 추가합니다.
- **긴 문단:** 길이별 표는 구간별 표본 수와 함께 해석합니다. 성능 차이만으로 문단 길이가 원인이라고 단정하지 않습니다.
- **전처리 제약:** 정답 구절이 창 안에 들어오고 길이 제한으로 복원 가능한지 먼저 확인합니다.

### 적용 범위와 다음 단계

KorQuAD 1.0은 문단 안에 답이 있는 위키백과 기반 추출형 질의응답입니다. 안내문·계약서 등 새로운 도메인에
적용하려면 해당 도메인의 별도 평가 자료가 필요합니다. 답이 없는 질문의 거절 기능은 별도 데이터와 평가로
검증해야 하며, 여러 문단을 포함한 문서는 문단 검색 단계를 추가해야 합니다. 직접 입력 예시 1개의 성공 여부는
시연 결과이며 일반화 성능을 입증하지 않습니다. 공개 검증의 오답을 보고 개선한 뒤 같은 자료로 다시 평가하면
최종 평가의 독립성이 약해지므로, 후속 실험은 내부 검증에서 설계하고 필요하면 새 평가 자료를 확보합니다.

참고: [KorQuAD 공식 소개](https://korquad.github.io/category/1.0_KOR.html),
[KLUE-RoBERTa 모델 카드](https://huggingface.co/klue/roberta-base),
[Hugging Face 질의응답 가이드](https://huggingface.co/docs/transformers/tasks/question_answering)


In [ ]:
def markdown_table(frame):
    # tabulate 등 추가 패키지 없이 보고서 표를 만듭니다.
    table = frame.reset_index()
    def format_value(value):
        text = f"{value:.2f}" if isinstance(value, (float, np.floating)) else str(value)
        return text.replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(map(str, table.columns)) + " |",
             "| " + " | ".join(["---"] * len(table.columns)) + " |"]
    lines.extend("| " + " | ".join(format_value(value) for value in row) + " |"
                 for row in table.itertuples(index=False, name=None))
    return "\n".join(lines)

mode_label = f"학습 자료 {TRAIN_FRACTION:.0%}, {NUM_EPOCHS} epoch"
report_parts = [
    f"# 한국어 문서 질의응답 실험 보고서 ({mode_label})",
    f"실행 ID: {RUN_ID} / 장치: {device_name} / seed: {SEED}",
    "## 데이터와 학습",
    f"내부 학습 {len(train_examples):,}개, 내부 검증 {len(selection_examples):,}개, "
    f"공개 검증 {len(final_examples):,}개를 사용했습니다. "
    f"내부 학습·검증은 기사 제목 단위로 분리했으며 학습 분할 {source_train_size:,}개 중 {TRAIN_FRACTION:.0%}를 추출했습니다. 검증 자료는 모두 유지했습니다.",
    f"모델: {MODEL_NAME}. 학습률 {training_args.learning_rate}, "
    f"epoch {training_args.num_train_epochs:g}, "
    f"학습 배치 {training_args.per_device_train_batch_size}, "
    f"gradient accumulation {training_args.gradient_accumulation_steps}.",
    f"내부 검증 기준 최적 loss: {trainer.state.best_metric:.4f}. "
    f"선택된 체크포인트: `{Path(trainer.state.best_model_checkpoint).name}`.",
    "학습·검증 곡선과 학습 기록은 앞의 셀 출력에 포함되어 있습니다. "
    "loss는 최적화 진단값이며 실제 답변 정확도는 아래 EM·글자 F1으로 평가합니다.",
    "## 공개 검증 성능",
    f"정규화 EM **{metrics['normalized_em']:.2f}**, "
    f"글자 F1 **{metrics['character_f1']:.2f}** (0~100, {metrics['examples']:,}개). "
    "공식 KorQuAD 리더보드 점수가 아닌 자체 지표입니다.",
    "### 문단 길이별 결과", markdown_table(by_length),
    "구간별 건수가 달라 점수 차이만으로 길이의 인과적 영향을 단정할 수 없습니다.",
    "## 오답 분석",
    f"EM 기준 오답 {len(error_df):,}개 / 전체 {len(result_df):,}개. "
    "아래 유형은 규칙으로 분류한 관찰 결과이며 원인에 대한 확정 진단은 아닙니다.",
    markdown_table(error_summary),
    case_analysis_markdown,
    "## 직접 작성한 문단 시연",
    f"<p>질문: {html.escape(new_question)}<br>"
    f"예측: {html.escape(answer['answer'])}<br>"
    f"기대 구절: {html.escape(demo_result['expected_answer'])}<br>"
    f"정규화 EM: {demo_em * 100:.2f}, 글자 F1: {demo_f1 * 100:.2f}</p>",
    "시연은 위키백과와 다른 문체의 단일 사례이며 서비스 성능을 보장하지 않습니다.",
    "## 한계와 후속 실험",
    "답이 없는 질문 거절, 새로운 도메인, 여러 문단 검색은 별도 평가가 필요합니다. "
    "오답 유형별 제안은 내부 검증에서 비교하고 공개 검증 점수로 설정을 선택하지 않습니다.",
    "실행 설정과 패키지 버전은 `run_config.json`에 기록했습니다.",
]
analysis_report = "\n\n".join(report_parts)
display(Markdown(analysis_report))


print("지정한 학습 자료 20%로 5 epoch 학습·평가·오류 분석을 완료했습니다. 출력이 포함된 이 노트북 하나를 저장하세요.")
